In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# verify secrets load
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
github_pat = secrets.get_secret("GITHUB_PAT")
print("WANDB_API_KEY length:", len(wandb_key))
print("GITHUB_PAT length:", len(github_pat))

WANDB_API_KEY length: 86
GITHUB_PAT length: 93


In [3]:
# verify W&B login
import wandb
wandb.login(key=wandb_key)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [4]:
# test W&B init in target project
run = wandb.init(
    project="23f2001849-t22026",
    entity="23f2001849-dl-genai-project",
    name="m2_preflight_check",
    tags=["preflight", "test"],
    config={"phase": "block_A_preflight"},
)
print(run.url)
run.finish()

https://wandb.ai/23f2001849-dl-genai-project/23f2001849-t22026/runs/jjewicir


In [5]:
# training imports and runtime check
import torch
import transformers
import re
from pathlib import Path
from sklearn.model_selection import GroupKFold
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

torch: 2.10.0+cu128 cuda: True
transformers: 5.0.0


In [6]:
# inline stem normalization
_PREFIXES = [
    "Pick the best possible answer:",
    "Select the most accurate option:",
    "Determine the correct option:",
    "Identify the correct statement:",
    "Choose the correct answer:",
    "Which of the following is correct?",
]
_SUFFIXES = [
    "among the listed options.",
    "based on the given context.",
    "from the following choices.",
    "carefully.",
]
_PREFIX_RE = re.compile(r"^\s*(?:" + "|".join(re.escape(p) for p in _PREFIXES) + r")\s*")
_SUFFIX_RE = re.compile(r"\s*(?:" + "|".join(re.escape(s) for s in _SUFFIXES) + r")\s*$")
_WS_RE = re.compile(r"\s+")

def normalize_stem(text):
    if not isinstance(text, str):
        return text
    s = text
    while True:
        before = s
        s = _PREFIX_RE.sub("", s)
        s = _SUFFIX_RE.sub("", s)
        if s == before:
            break
    s = _WS_RE.sub(" ", s).strip()
    return s

In [7]:
# load train, normalize, recompute folds
import pandas as pd
comp_dir = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
train = pd.read_csv(comp_dir / "train.csv")
train["stem"] = train["prompt"].apply(normalize_stem)

N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
train["fold"] = -1
for fold_idx, (_, val_idx) in enumerate(gkf.split(train, groups=train["stem"])):
    train.loc[val_idx, "fold"] = fold_idx
print(train["fold"].value_counts().sort_index())
assert (train["fold"].value_counts() == 400).all(), "fold sizes do not match local"

fold
0    400
1    400
2    400
3    400
4    400
Name: count, dtype: int64


In [8]:
# multiple-choice dataset
OPT_COLS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {c: i for i, c in enumerate(OPT_COLS)}

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encodings = self.tokenizer(
            [row["stem"]] * 5,
            [row[c] for c in OPT_COLS],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": encodings["input_ids"],
            "attention_mask": encodings["attention_mask"],
            "label": LABEL2IDX[row["answer"]],
        }

In [9]:
# load tokenizer and model, smoke-test forward pass
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to("cuda")
model.train()

smoke_df = train[train["fold"] != 0].head(4).reset_index(drop=True)
smoke_ds = MCQDataset(smoke_df, tokenizer, max_len=256)
smoke_loader = DataLoader(smoke_ds, batch_size=2, shuffle=False)

batch = next(iter(smoke_loader))
input_ids = batch["input_ids"].to("cuda")
attention_mask = batch["attention_mask"].to("cuda")
labels = batch["label"].to("cuda")
print("input_ids shape:", input_ids.shape)

out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
print("loss:", out.loss.item())
print("logits shape:", out.logits.shape)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


input_ids shape: torch.Size([2, 5, 256])
loss: 1.6395952701568604
logits shape: torch.Size([2, 5])


In [10]:
# end-to-end training step with W&B logging, gradient clipping, NaN guard
import math
import wandb

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6, weight_decay=0.01)

run = wandb.init(
    project="23f2001849-t22026",
    entity="23f2001849-dl-genai-project",
    name="m2_smoke_fold0_v2",
    group="m2_deberta_v3_base",
    tags=["m2", "deberta-v3-base", "smoke"],
    config={
        "phase": "block_C_smoke",
        "fold": 0,
        "max_len": 256,
        "bs": 2,
        "lr": 2e-5,
        "eps": 1e-6,
        "weight_decay": 0.01,
        "grad_clip": 1.0,
    },
)

for step in range(5):
    optimizer.zero_grad()
    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    out.loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    loss_val = out.loss.item()
    print(f"step {step}: loss={loss_val:.4f} grad_norm={grad_norm:.4f}")
    wandb.log({"train/loss": loss_val, "train/grad_norm": grad_norm.item(), "step": step})
    assert not math.isnan(loss_val), f"NaN at step {step}"

print(run.url)
run.finish()

step 0: loss=1.5633 grad_norm=6.7409
step 1: loss=1.6002 grad_norm=2.2231
step 2: loss=1.6276 grad_norm=3.1038
step 3: loss=1.5731 grad_norm=2.2944
step 4: loss=1.6232 grad_norm=4.5053
https://wandb.ai/23f2001849-dl-genai-project/23f2001849-t22026/runs/vs26nd8j


step,▁▃▅▆█
train/grad_norm,█▁▂▁▅
train/loss,▁▅█▂█
step,4
train/grad_norm,4.50526
train/loss,1.62316


In [11]:
# verify checkpoint save and reload
ckpt_path = Path("/kaggle/working/smoke_checkpoint.pt")
torch.save({"model_state": model.state_dict()}, ckpt_path)
size_mb = ckpt_path.stat().st_size / 1e6
print(f"saved {ckpt_path}, size: {size_mb:.1f} MB")

state = torch.load(ckpt_path, map_location="cuda")
print("state dict keys:", len(state["model_state"]))

saved /kaggle/working/smoke_checkpoint.pt, size: 498.7 MB
state dict keys: 201


In [12]:
# fold 0 training config
import random

SEED = 42
FOLD = 0
MAX_LEN = 256
BATCH_SIZE = 4
EPOCHS = 2
LR = 1e-5
EPS = 1e-6
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
WARMUP_RATIO = 0.1

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("config set, seed", SEED)

config set, seed 42


In [13]:
# fold 0 train and val splits, dataloaders
train_df = train[train["fold"] != FOLD].reset_index(drop=True)
val_df = train[train["fold"] == FOLD].reset_index(drop=True)
print("train rows:", len(train_df), "val rows:", len(val_df))

train_ds = MCQDataset(train_df, tokenizer, max_len=MAX_LEN)
val_ds = MCQDataset(val_df, tokenizer, max_len=MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("train batches:", len(train_loader), "val batches:", len(val_loader))

train rows: 1600 val rows: 400
train batches: 400 val batches: 100


In [14]:
# MAP@3 over a batch of (logits, labels)
def compute_map3(logits, labels):
    top3 = logits.argsort(dim=1, descending=True)[:, :3]
    ap = torch.zeros(len(labels))
    for k in range(3):
        match = (top3[:, k] == labels)
        ap += match.float() / (k + 1)
    return ap.mean().item()

In [15]:
# 5-fold cv training, locked config
import time
from transformers import get_linear_schedule_with_warmup

N_FOLDS = 5
fold_scores = []

def freeze_lower(m):
    for name, p in m.named_parameters():
        if name.startswith("roberta.embeddings") or any(name.startswith(f"roberta.encoder.layer.{i}.") for i in range(6)):
            p.requires_grad = False
    return [p for p in m.parameters() if p.requires_grad]

for fold in range(N_FOLDS):
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    train_df = train[train["fold"] != fold].reset_index(drop=True)
    val_df = train[train["fold"] == fold].reset_index(drop=True)
    train_ds = MCQDataset(train_df, tokenizer, max_len=MAX_LEN)
    val_ds = MCQDataset(val_df, tokenizer, max_len=MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to("cuda")
    trainable = freeze_lower(model)
    optimizer = torch.optim.AdamW(trainable, lr=LR, eps=EPS, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    run = wandb.init(
        project="23f2001849-t22026",
        entity="23f2001849-dl-genai-project",
        name=f"m2_roberta_fold{fold}_reg",
        group="m2_roberta_base_cv",
        tags=["m2", "roberta-base", "workstream-b", "reg", "cv"],
        config={
            "fold": fold, "max_len": MAX_LEN, "batch_size": BATCH_SIZE, "epochs": EPOCHS,
            "lr": LR, "eps": EPS, "weight_decay": WEIGHT_DECAY, "grad_clip": GRAD_CLIP,
            "warmup_ratio": WARMUP_RATIO, "seed": SEED, "model_name": MODEL_NAME,
            "frozen": "embeddings+layers0-5",
        },
    )

    best_val_map3 = -1.0
    best_ckpt_path = Path(f"/kaggle/working/m2_roberta_fold{fold}_best.pt")
    t_start = time.time()

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []
        for batch in train_loader:
            input_ids = batch["input_ids"].to("cuda")
            attention_mask = batch["attention_mask"].to("cuda")
            labels = batch["label"].to("cuda")
            optimizer.zero_grad()
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            out.loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(trainable, max_norm=GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            train_losses.append(out.loss.item())
            wandb.log({"train/loss": out.loss.item(), "train/grad_norm": grad_norm.item(), "lr": optimizer.param_groups[0]["lr"]})

        model.eval()
        val_logits_all, val_labels_all, val_losses = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to("cuda")
                attention_mask = batch["attention_mask"].to("cuda")
                labels = batch["label"].to("cuda")
                out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                val_logits_all.append(out.logits.cpu())
                val_labels_all.append(labels.cpu())
                val_losses.append(out.loss.item())
        val_logits = torch.cat(val_logits_all)
        val_labels = torch.cat(val_labels_all)
        val_map3 = compute_map3(val_logits, val_labels)
        val_loss = float(np.mean(val_losses))
        train_loss = float(np.mean(train_losses))
        logits_std = float(val_logits.std(dim=1).mean())
        elapsed = time.time() - t_start
        print(f"fold {fold} epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_map3={val_map3:.4f} logits_std={logits_std:.4f} elapsed={elapsed:.1f}s")
        wandb.log({"epoch": epoch, "epoch/train_loss": train_loss, "epoch/val_loss": val_loss, "epoch/val_map3": val_map3, "epoch/val_logits_std": logits_std})

        if val_map3 > best_val_map3:
            best_val_map3 = val_map3
            torch.save({"model_state": model.state_dict(), "val_map3": val_map3, "epoch": epoch, "fold": fold}, best_ckpt_path)

    wandb.run.summary["best_val_map3"] = best_val_map3
    run.finish()
    fold_scores.append(best_val_map3)
    print(f"fold {fold} best_val_map3={best_val_map3:.4f}")

print("\nper-fold:", [round(s, 4) for s in fold_scores])
print(f"cv mean val_map3={np.mean(fold_scores):.4f} std={np.std(fold_scores):.4f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 0 epoch 0: train_loss=1.5396 val_loss=1.4986 val_map3=0.4871 logits_std=0.3271 elapsed=232.6s
fold 0 epoch 1: train_loss=1.0862 val_loss=1.6029 val_map3=0.4742 logits_std=0.4555 elapsed=469.0s


epoch,▁█
epoch/train_loss,█▁
epoch/val_logits_std,▁█
epoch/val_loss,▁█
epoch/val_map3,█▁
lr,▃▆█████▇▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▂▂▁
train/grad_norm,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▅▂▂▄█▄▅▃▅▆▅▆▄▂▃▅▃▆▃▂▃
train/loss,█▇███▇▇▇▆▆▇▅▅▆▅█▄▅▆▅▃▇▇▂▆▇▅▆▄▁▄▄▂▅▁▃▄▃▅▂
best_val_map3,0.48708
epoch,1
epoch/train_loss,1.08624


fold 0 best_val_map3=0.4871


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.bias       | MISSING    | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold 4 epoch 0: train_loss=1.5080 val_loss=1.4020 val_map3=0.5629 logits_std=0.4920 elapsed=236.1s
fold 4 epoch 1: train_loss=0.9441 val_loss=1.4267 val_map3=0.5713 logits_std=0.7352 elapsed=472.1s


epoch,▁█
epoch/train_loss,█▁
epoch/val_logits_std,▁█
epoch/val_loss,▁█
epoch/val_map3,▁█
lr,▂▂▄██████▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁
train/grad_norm,▁▁▁▁▁▁▁▂▁▂▁▁▂▂▂▃▂▃▃▃▇▃▄▅▃▅▆▅▆▄▆▅▆▇▅▂▅█▇▇
train/loss,▇▇▇▇▇▇▇▇▇█▇▇▆▇▆▇▅▅▆▅▆▂▅█▅▅▃▅▅▄▁▅▃▁▂▁▆▅▁▄
best_val_map3,0.57125
epoch,1
epoch/train_loss,0.94409


fold 4 best_val_map3=0.5713

per-fold: [0.4871, 0.5383, 0.5608, 0.4638, 0.5713]
cv mean val_map3=0.5243 std=0.0419


In [16]:
# diagnostic: inspect tokenized inputs and model output for one val sample
sample_ds = MCQDataset(val_df.iloc[[0]].reset_index(drop=True), tokenizer, max_len=MAX_LEN)
sample_batch = next(iter(DataLoader(sample_ds, batch_size=1)))

print("input_ids shape:", sample_batch["input_ids"].shape)
print()
print("token differences between option k and option 0:")
for k in range(1, 5):
    diff = (sample_batch["input_ids"][0, k] != sample_batch["input_ids"][0, 0]).sum().item()
    print(f"  option {k} differs from option 0 in {diff} positions")

print()
print("decoded option 0 (first 250 chars):")
print(tokenizer.decode(sample_batch["input_ids"][0, 0], skip_special_tokens=False)[:250])
print()
print("decoded option 1 (first 250 chars):")
print(tokenizer.decode(sample_batch["input_ids"][0, 1], skip_special_tokens=False)[:250])

model.eval()
with torch.no_grad():
    out = model(
        input_ids=sample_batch["input_ids"].to("cuda"),
        attention_mask=sample_batch["attention_mask"].to("cuda"),
    )
print()
print("raw logits:", out.logits.squeeze().tolist())
print("logits std:", out.logits.std().item())
print("logits all equal?", (out.logits[0] == out.logits[0, 0]).all().item())

input_ids shape: torch.Size([1, 5, 256])

token differences between option k and option 0:
  option 1 differs from option 0 in 22 positions
  option 2 differs from option 0 in 22 positions
  option 3 differs from option 0 in 22 positions
  option 4 differs from option 0 in 27 positions

decoded option 0 (first 250 chars):
<s>What is the interpretation of supersymmetry in stochastic supersymmetric theory?</s></s>Supersymmetry is a type of hydromagnetic dynamo that arises when the magnetic field becomes strong enough to affect the fluid motions.</s><pad><pad><pad><pad><

decoded option 1 (first 250 chars):
<s>What is the interpretation of supersymmetry in stochastic supersymmetric theory?</s></s>Supersymmetry is a measure of the amplitude of the dynamo in the induction equation of the kinematic approximation.</s><pad><pad><pad><pad><pad><pad><pad><pad>

raw logits: [-2.1711394786834717, -2.162379264831543, -2.207371473312378, -2.1264824867248535, 3.571868419647217]
logits std: 2.56659102